In [8]:
# ============================================================
# 3D U-NET - TASK03 LIVER NPY DATASET
# FINAL CORRECTED SINGLE COLAB CELL
# ============================================================

!pip install -q kagglehub scipy tqdm


# ============================
# IMPORTS
# ============================

import os
import numpy as np

import kagglehub

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader, random_split

from scipy.ndimage import zoom

from tqdm import tqdm



# ============================
# DATASET DOWNLOAD
# ============================

path = kagglehub.dataset_download(
    "zeynepzelk/task03-liver-npy-dataset"
)


print("Dataset path:")
print(path)



IMG_DIR=os.path.join(
    path,
    "image"
)


MASK_DIR=os.path.join(
    path,
    "liverMask"
)



print(
    "Images:",
    len(os.listdir(IMG_DIR))
)

print(
    "Masks:",
    len(os.listdir(MASK_DIR))
)



# ============================
# PARAMETERS
# ============================

TARGET_SIZE=(64,128,128)

BATCH_SIZE=1

EPOCHS=5


LR=1e-4



# ============================
# PREPROCESSING
# ============================

def normalize(x):

    x=np.clip(
        x,
        -200,
        250
    )

    x=(x-x.min())/(x.max()-x.min()+1e-8)

    return x




def resize_volume(x):

    factors=(

        TARGET_SIZE[0]/x.shape[0],
        TARGET_SIZE[1]/x.shape[1],
        TARGET_SIZE[2]/x.shape[2]

    )

    return zoom(
        x,
        factors,
        order=1
    )



# ============================
# DATASET CORRECTED
# ============================

class LiverDataset(Dataset):

    def __init__(self,img_dir,mask_dir):

        self.img_dir=img_dir
        self.mask_dir=mask_dir


        images=sorted(
            [
                f for f in os.listdir(img_dir)
                if f.endswith(".npy")
            ]
        )


        self.masks=sorted(
            [
                f for f in os.listdir(mask_dir)
                if f.endswith(".npy")
            ]
        )



        self.files=[]


        # Keep only images with masks

        for img in images:


            base=img.replace(
                "_img.npy",
                ""
            )


            exists=False


            for m in self.masks:

                if base in m:

                    exists=True
                    break



            if exists:

                self.files.append(img)



        print(
            "Valid image-mask pairs:",
            len(self.files)
        )



    def __len__(self):

        return len(self.files)



    def __getitem__(self,index):


        img_file=self.files[index]


        img=np.load(
            os.path.join(
                self.img_dir,
                img_file
            )
        )



        base=img_file.replace(
            "_img.npy",
            ""
        )



        mask_file=[

            m for m in self.masks

            if base in m

        ][0]



        mask=np.load(
            os.path.join(
                self.mask_dir,
                mask_file
            )
        )



        img=normalize(img)


        img=resize_volume(img)

        mask=resize_volume(mask)



        mask=(mask>0).astype(
            np.float32
        )



        img=torch.tensor(
            img,
            dtype=torch.float32
        ).unsqueeze(0)



        mask=torch.tensor(
            mask,
            dtype=torch.float32
        ).unsqueeze(0)



        return img,mask





# ============================
# LOAD DATA
# ============================

dataset=LiverDataset(
    IMG_DIR,
    MASK_DIR
)



train_size=int(
    0.8*len(dataset)
)


val_size=len(dataset)-train_size



train_dataset,val_dataset=random_split(

    dataset,

    [
        train_size,
        val_size
    ]

)



train_loader=DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True
)



val_loader=DataLoader(
    val_dataset,
    batch_size=1
)



print(
    "Train:",
    len(train_dataset)
)


print(
    "Validation:",
    len(val_dataset)
)



# ============================
# 3D U-NET
# ============================


class ConvBlock(nn.Module):

    def __init__(self,in_c,out_c):

        super().__init__()


        self.block=nn.Sequential(

            nn.Conv3d(
                in_c,
                out_c,
                3,
                padding=1
            ),

            nn.BatchNorm3d(out_c),

            nn.ReLU(inplace=True),


            nn.Conv3d(
                out_c,
                out_c,
                3,
                padding=1
            ),

            nn.BatchNorm3d(out_c),

            nn.ReLU(inplace=True)

        )



    def forward(self,x):

        return self.block(x)




class UNet3D(nn.Module):

    def __init__(self):

        super().__init__()



        self.e1=ConvBlock(1,32)

        self.e2=ConvBlock(32,64)

        self.e3=ConvBlock(64,128)



        self.pool=nn.MaxPool3d(2)



        self.b=ConvBlock(
            128,
            256
        )



        self.up3=nn.ConvTranspose3d(
            256,
            128,
            2,
            2
        )


        self.d3=ConvBlock(
            256,
            128
        )



        self.up2=nn.ConvTranspose3d(
            128,
            64,
            2,
            2
        )


        self.d2=ConvBlock(
            128,
            64
        )



        self.up1=nn.ConvTranspose3d(
            64,
            32,
            2,
            2
        )


        self.d1=ConvBlock(
            64,
            32
        )



        self.out=nn.Conv3d(
            32,
            1,
            1
        )



    def forward(self,x):


        e1=self.e1(x)


        e2=self.e2(
            self.pool(e1)
        )


        e3=self.e3(
            self.pool(e2)
        )



        b=self.b(
            self.pool(e3)
        )



        d3=self.up3(b)


        d3=torch.cat(
            [
                d3,
                e3
            ],
            dim=1
        )


        d3=self.d3(d3)




        d2=self.up2(d3)


        d2=torch.cat(
            [
                d2,
                e2
            ],
            dim=1
        )


        d2=self.d2(d2)




        d1=self.up1(d2)



        d1=torch.cat(
            [
                d1,
                e1
            ],
            dim=1
        )


        d1=self.d1(d1)



        return torch.sigmoid(
            self.out(d1)
        )




# ============================
# DICE LOSS
# ============================

def dice_loss(pred,target):

    smooth=1e-5


    intersection=(pred*target).sum()


    dice=(2*intersection+smooth)/(
        pred.sum()+target.sum()+smooth
    )


    return 1-dice



# ============================
# TRAINING
# ============================


device="cuda" if torch.cuda.is_available() else "cpu"


print(
    "Device:",
    device
)



model=UNet3D().to(device)



optimizer=torch.optim.Adam(
    model.parameters(),
    lr=LR
)




for epoch in range(EPOCHS):


    model.train()


    loss_epoch=0



    bar=tqdm(train_loader)



    for img,mask in bar:


        img=img.to(device)

        mask=mask.to(device)



        pred=model(img)



        loss=dice_loss(
            pred,
            mask
        )



        optimizer.zero_grad()

        loss.backward()

        optimizer.step()



        loss_epoch+=loss.item()



        bar.set_description(
            f"Epoch {epoch+1}/{EPOCHS}"
        )



    print(
        "Loss:",
        loss_epoch/len(train_loader)
    )



# ============================
# VALIDATION
# ============================


model.eval()


dice_values=[]



with torch.no_grad():


    for img,mask in val_loader:


        img=img.to(device)

        mask=mask.to(device)



        pred=model(img)


        pred=(pred>0.5).float()



        inter=(pred*mask).sum()



        dice=(2*inter)/(
            pred.sum()+mask.sum()+1e-5
        )



        dice_values.append(
            dice.item()
        )



print(
    "Validation Dice:",
    np.mean(dice_values)
)



# ============================
# SAVE MODEL
# ============================


torch.save(
    model.state_dict(),
    "3D_UNet_Liver.pth"
)



print(
    "Saved model: 3D_UNet_Liver.pth"
)

Using Colab cache for faster access to the 'task03-liver-npy-dataset' dataset.
Dataset path:
/kaggle/input/task03-liver-npy-dataset
Images: 131
Masks: 130
Valid image-mask pairs: 130
Train: 104
Validation: 26
Device: cuda


Epoch 1/5: 100%|██████████| 104/104 [02:59<00:00,  1.72s/it]


Loss: 0.8608882742432448


Epoch 2/5: 100%|██████████| 104/104 [02:45<00:00,  1.60s/it]


Loss: 0.8387872169797237


...

Loss: 0.826294964322677


Epoch 4/5: 100%|██████████| 104/104 [02:39<00:00,  1.54s/it]


Loss: 0.8118877479663262


Epoch 5/5: 100%|██████████| 104/104 [02:40<00:00,  1.54s/it]


Loss: 0.7936921681349094
Validation Dice: 0.9589952826499939
Saved model: 3D_UNet_Liver.pth


In [2]:
import os

print("Images:")
print(sorted(os.listdir(IMG_DIR))[:10])

print("\nMasks:")
print(sorted(os.listdir(MASK_DIR))[:10])

Images:
['liver_0_img.npy', 'liver_100_img.npy', 'liver_101_img.npy', 'liver_102_img.npy', 'liver_103_img.npy', 'liver_104_img.npy', 'liver_105_img.npy', 'liver_106_img.npy', 'liver_107_img.npy', 'liver_108_img.npy']

Masks:
['liver_0_liverMask.npy', 'liver_100_liverMask.npy', 'liver_101_liverMask.npy', 'liver_102_liverMask.npy', 'liver_103_liverMask.npy', 'liver_104_liverMask.npy', 'liver_105_liverMask.npy', 'liver_106_liverMask.npy', 'liver_107_liverMask.npy', 'liver_108_liverMask.npy']


In [3]:
DATA_DIR = "/kaggle/input/liver-cancer-multiclass-dataset/Liver_Dataset/Liver_Dataset"